# 01 — New Mexico Landsat Time Series (2023–2025)

Generates annual field boundary polygons using Landsat 8/9 imagery and the **Delineate-Anything** engine. Uses NMOSE (New Mexico Office of the State Engineer) WUCB agricultural polygon boundaries for fine-tuning and evaluation.

This notebook runs a 3-year subset (2023–2025) suitable for interactive use. The corresponding Python script (`01_new_mexico_landsat_timeseries.py`) runs the full 40-year range (1985–2025) and is better suited for HPC/batch execution.

**Estimated runtime:** ~30–60 minutes (3 years of GEE composite + GPU inference + fine-tuning).

**Prerequisites:**
```bash
pip install agribound[gee,delineate-anything]
agribound auth --project YOUR_GEE_PROJECT
```

## Configuration

In [ ]:
import json
from pathlib import Path

import geopandas as gpd

import agribound
from agribound.evaluate import evaluate

# --- Configuration ---
GEE_PROJECT = "YOUR_GEE_PROJECT"  # <-- Replace with your GEE project ID

NMOSE_SHAPEFILE = "../NMOSE Field Boundaries/WUCB ag polys.shp"
OUTPUT_DIR = Path("../../outputs/new_mexico_timeseries")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CRS = "EPSG:26913"  # Match NMOSE reference CRS (NAD83 / UTM zone 13N)

YEARS = range(2023, 2026)
SOURCE = "landsat"
ENGINE = "delineate-anything"
FINE_TUNE = True

## Derive Study Area from NMOSE Shapefile Bounds

In [ ]:
ref_gdf = gpd.read_file(NMOSE_SHAPEFILE)
print(f"{len(ref_gdf)} reference field polygons loaded")

# Reproject to WGS84 for GeoJSON (shapefile may be in a projected CRS)
ref_4326 = ref_gdf.to_crs(epsg=4326)
bounds = ref_4326.total_bounds  # [minx, miny, maxx, maxy]
bbox_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [bounds[0], bounds[1]],
                        [bounds[2], bounds[1]],
                        [bounds[2], bounds[3]],
                        [bounds[0], bounds[3]],
                        [bounds[0], bounds[1]],
                    ]
                ],
            },
            "properties": {"name": "NMOSE WUCB Study Area"},
        }
    ],
}
study_area = str(OUTPUT_DIR / "nm_study_area.geojson")
with open(study_area, "w") as f:
    json.dump(bbox_geojson, f)

print(f"Study area saved to {study_area}")
print(f"Bounds (WGS84): {bounds}")

## Phase 1: Fine-tune on NMOSE Reference Boundaries

Fine-tunes the Delineate-Anything engine on the NMOSE WUCB reference polygons using a 2024 Landsat composite. The fine-tuned checkpoint is saved to disk and reused for all subsequent years.

In [ ]:
checkpoint_path = None

if FINE_TUNE:
    finetune_year = 2024
    finetune_output = OUTPUT_DIR / f"fields_finetuned_{finetune_year}.gpkg"
    checkpoint_dir = OUTPUT_DIR / "checkpoints"

    existing_ckpts = list(checkpoint_dir.glob("*.pt")) if checkpoint_dir.exists() else []
    if existing_ckpts:
        checkpoint_path = str(existing_ckpts[0])
        print(f"Reusing existing checkpoint: {checkpoint_path}")
    else:
        gdf_ft = agribound.delineate(
            study_area=study_area,
            source=SOURCE,
            year=finetune_year,
            engine=ENGINE,
            output_path=str(finetune_output),
            gee_project=GEE_PROJECT,
            composite_method="median",
            cloud_cover_max=20,
            min_area=2500,
            simplify=2.0,
            device="auto",
            reference_boundaries=NMOSE_SHAPEFILE,
            fine_tune=True,
        )
        # Reproject to match NMOSE reference CRS
        if gdf_ft.crs is not None and str(gdf_ft.crs) != OUTPUT_CRS:
            gdf_ft = gdf_ft.to_crs(OUTPUT_CRS)
            gdf_ft.to_file(finetune_output, driver="GPKG")
        print(f"Fine-tuned model produced {len(gdf_ft)} fields for {finetune_year}")

        metrics = evaluate(gdf_ft, ref_gdf)
        print(f"IoU:       {metrics['iou_mean']:.3f}")
        print(f"Precision: {metrics['precision']:.3f}")
        print(f"Recall:    {metrics['recall']:.3f}")
        print(f"F1:        {metrics['f1']:.3f}")

        new_ckpts = list(checkpoint_dir.glob("*.pt")) if checkpoint_dir.exists() else []
        if new_ckpts:
            checkpoint_path = str(new_ckpts[0])
            print(f"Checkpoint saved: {checkpoint_path}")

## Phase 2: Annual Field Boundary Delineation (2023–2025)

Runs inference for each year using the fine-tuned checkpoint (if available). Each year's composite is downloaded from GEE, delineated, and evaluated against the NMOSE reference.

In [ ]:
engine_params = {}
if checkpoint_path:
    engine_params["checkpoint_path"] = checkpoint_path
    print(f"Using fine-tuned checkpoint: {checkpoint_path}")

all_results = {}

for year in YEARS:
    print(f"Processing year {year}...", end=" ")
    output_path = OUTPUT_DIR / f"fields_landsat_{year}.gpkg"

    if output_path.exists():
        print("already exists, skipping.")
        all_results[year] = gpd.read_file(output_path)
        continue

    try:
        gdf = agribound.delineate(
            study_area=study_area,
            source=SOURCE,
            year=year,
            engine=ENGINE,
            output_path=str(output_path),
            gee_project=GEE_PROJECT,
            composite_method="median",
            cloud_cover_max=20,
            min_area=2500,
            simplify=2.0,
            device="auto",
            reference_boundaries=NMOSE_SHAPEFILE,
            fine_tune=False,
            engine_params=engine_params,
        )
        # Reproject to match NMOSE reference CRS
        if gdf.crs is not None and str(gdf.crs) != OUTPUT_CRS:
            gdf = gdf.to_crs(OUTPUT_CRS)
            gdf.to_file(output_path, driver="GPKG")
        all_results[year] = gdf
        print(f"{len(gdf)} fields")
    except Exception as exc:
        print(f"FAILED: {exc}")

## Phase 3: Summary Statistics

In [ ]:
print(f"{'Year':<6} {'Fields':>6} {'Area (ha)':>12} {'F1':>6} {'IoU':>6}")
print(f"{'-' * 6} {'-' * 6} {'-' * 12} {'-' * 6} {'-' * 6}")

for year, gdf in sorted(all_results.items()):
    area_ha = gdf["metrics:area"].sum() / 10000 if "metrics:area" in gdf.columns else 0
    f1 = iou = ""
    if hasattr(gdf, "attrs") and "evaluation_metrics" in gdf.attrs:
        m = gdf.attrs["evaluation_metrics"]
        f1 = f"{m['f1']:.3f}"
        iou = f"{m['iou_mean']:.3f}"
    print(f"{year:<6} {len(gdf):>6} {area_ha:>12,.1f} {f1:>6} {iou:>6}")

## Phase 4: Visualization

Interactive maps showing predicted boundaries vs NMOSE reference and a multi-year comparison.

In [ ]:
from agribound.visualize import show_comparison

if all_results:
    latest_year = max(all_results.keys())
    m = show_comparison(
        [all_results[latest_year], ref_gdf],
        labels=[f"Predicted ({latest_year})", "NMOSE Reference"],
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_predicted_vs_reference.html"),
    )
    m

In [ ]:
# Multi-year comparison (2023–2025)
boundaries_list = [all_results[y] for y in sorted(all_results.keys())]
labels = [str(y) for y in sorted(all_results.keys())]

if len(boundaries_list) >= 2:
    m_compare = show_comparison(
        boundaries_list,
        labels=labels,
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_timeseries_comparison.html"),
    )
    m_compare

In [ ]:
# Latest year standalone map
if all_results:
    m_latest = agribound.show_boundaries(
        all_results[max(all_results.keys())],
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_latest.html"),
    )
    m_latest